# 从零复现 DDPM：噪声日程、Tiny U-Net 与反向后验

本 Notebook 不使用 `diffusers`、现成 U-Net 或预训练权重。我们用基础 PyTorch 手写 sinusoidal timestep embedding、time-conditioned `ResBlock`、`TinyUNet.forward`、线性 $\beta_t$ schedule、前向加噪 $q(x_t\mid x_0)$、epsilon prediction loss，以及带正确 posterior mean/variance 的逐步反向采样。

重点验证公式和工程边界：同一显式 generator 必须复现噪声与样本；$t=0$ 不得再注入随机噪声；clipping 必须作用于预测的 $x_0$ 并重新计算 posterior mean；训练/validation/test 随机流要分开；制品必须绑定 schedule、数据范围和权重摘要。

全部数据离线合成、固定 seed、CPU 单线程。$8\times8$ 微型图案只验证扩散计算图，不代表现代图像生成质量。

## 1. 两个马尔可夫过程

前向过程逐步加入高斯噪声：

$$q(x_t\mid x_{t-1})=\mathcal N(\sqrt{\alpha_t}x_{t-1},\beta_tI),\quad \alpha_t=1-\beta_t.$$

利用重参数化可以一步得到任意时刻：

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\qquad
\bar\alpha_t=\prod_{s=0}^{t}\alpha_s.$$

模型学习 $\epsilon_\theta(x_t,t)$。采样从 $x_T\sim\mathcal N(0,I)$ 开始，按 $t=T-1,\ldots,0$ 迭代近似 $p_\theta(x_{t-1}\mid x_t)$。训练和采样必须使用同一个 schedule 定义。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from hashlib import sha256
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 360728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. 线性 beta schedule 与所有派生系数

教学版使用 32 步线性 $\beta$，范围远大于常见千步 DDPM，是为了让微型采样在 CPU 快速接近纯噪声。改变步数或端点会改变任务，属于模型制品的一部分。

真实后验方差为

$$\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t},$$

其中定义 $\bar\alpha_{-1}=1$，所以 $\tilde\beta_0=0$。实现不能为了数值方便把 $t=0$ 方差 clamp 成正数后继续采样。

In [ ]:
class DiffusionSchedule(nn.Module):
    def __init__(self, timesteps=32, beta_start=1e-4, beta_end=0.18):
        super().__init__()
        if timesteps < 2 or not 0 < beta_start < beta_end < 1:
            raise ValueError("invalid diffusion schedule")
        betas = torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        alpha_bars_prev = torch.cat([torch.ones(1), alpha_bars[:-1]])
        posterior_variance = betas * (1 - alpha_bars_prev) / (1 - alpha_bars)
        posterior_mean_coef1 = betas * alpha_bars_prev.sqrt() / (1 - alpha_bars)
        posterior_mean_coef2 = (1 - alpha_bars_prev) * alphas.sqrt() / (1 - alpha_bars)
        self.timesteps = int(timesteps)
        self.beta_start, self.beta_end = float(beta_start), float(beta_end)
        for name, tensor in {
            "betas": betas, "alphas": alphas, "alpha_bars": alpha_bars,
            "alpha_bars_prev": alpha_bars_prev,
            "posterior_variance": posterior_variance,
            "posterior_mean_coef1": posterior_mean_coef1,
            "posterior_mean_coef2": posterior_mean_coef2,
        }.items():
            self.register_buffer(name, tensor)

    def forward(self, t):
        if t.dtype != torch.long or t.ndim != 1:
            raise ValueError("t must be int64 [B]")
        if (t < 0).any() or (t >= self.timesteps).any():
            raise ValueError("timestep out of range")
        return self.alpha_bars[t]

schedule36 = DiffusionSchedule().to(DEVICE)
assert schedule36.betas.shape == (32,)
assert torch.all((schedule36.betas > 0) & (schedule36.betas < 1))
assert torch.all(schedule36.alpha_bars[1:] < schedule36.alpha_bars[:-1])
assert schedule36.posterior_variance[0].item() == 0.0
direct_variance_t1 = (schedule36.betas[1] * (1 - schedule36.alpha_bars[0]) /
                      (1 - schedule36.alpha_bars[1]))
assert torch.allclose(schedule36.posterior_variance[1], direct_variance_t1)

## 3. 正弦 timestep embedding

网络若看不到 $t$，同一个 $x_t$ 无法区分轻噪声和重噪声阶段。对 embedding 维度 $d$，使用不同频率的 sin/cos：

$$e(t)=[\sin(t\omega_0),\ldots,\sin(t\omega_{d/2-1}),
\cos(t\omega_0),\ldots,\cos(t\omega_{d/2-1})].$$

它没有可训练参数，但后接 MLP 让网络学习适合当前任务的时间表示。奇数维会破坏对称拼接，接口直接拒绝。

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        if embedding_dim < 4 or embedding_dim % 2:
            raise ValueError("time embedding dimension must be even and >= 4")
        self.embedding_dim = int(embedding_dim)

    def forward(self, timesteps):
        if timesteps.ndim != 1:
            raise ValueError("timesteps must be [B]")
        half = self.embedding_dim // 2
        frequencies = torch.exp(-math.log(10000.0) *
                                torch.arange(half, device=timesteps.device) / max(half - 1, 1))
        angles = timesteps.float()[:, None] * frequencies[None, :]
        return torch.cat([angles.sin(), angles.cos()], dim=-1)

time_encoder = SinusoidalTimeEmbedding(16)
time_probe = time_encoder(torch.tensor([0, 1, 7], dtype=torch.long))
assert time_probe.shape == (3, 16)
assert torch.equal(time_probe[0, :8], torch.zeros(8))
assert torch.equal(time_probe[0, 8:], torch.ones(8))
assert not torch.equal(time_probe[1], time_probe[2])

## 4. time-conditioned ResBlock

每个 block 先处理图像特征，再把 time MLP 投影成 `[B,Cout,1,1]` 加入中间激活。输入输出通道不同时，skip path 使用 $1\times1$ projection；相同时使用恒等映射。

GroupNorm 不依赖 batch running statistics，适合扩散训练常见的小 batch。它仍要求 `num_channels` 能被 group 数整除，因此实现显式选择合法 group。

In [ ]:
def valid_groups(channels):
    return 4 if channels % 4 == 0 else 1

class TimeConditionedResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.in_channels, self.out_channels, self.time_dim = in_channels, out_channels, time_dim
        self.norm1 = nn.GroupNorm(valid_groups(in_channels), in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.time_projection = nn.Linear(time_dim, out_channels)
        self.norm2 = nn.GroupNorm(valid_groups(out_channels), out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, time_embedding):
        if x.ndim != 4 or x.shape[1] != self.in_channels:
            raise ValueError("ResBlock feature shape mismatch")
        if time_embedding.shape != (x.shape[0], self.time_dim):
            raise ValueError("ResBlock time embedding mismatch")
        hidden = self.conv1(F.silu(self.norm1(x)))
        hidden = hidden + self.time_projection(F.silu(time_embedding))[:, :, None, None]
        hidden = self.conv2(F.silu(self.norm2(hidden)))
        return self.skip(x) + hidden

resblock_probe = TimeConditionedResBlock(4, 8, 16)
resblock_x = torch.randn(2, 4, 8, 8, requires_grad=True)
resblock_t = torch.randn(2, 16, requires_grad=True)
resblock_y = resblock_probe(resblock_x, resblock_t)
resblock_y.mean().backward()
assert resblock_y.shape == (2, 8, 8, 8)
assert resblock_x.grad is not None and float(resblock_x.grad.norm()) > 0
assert resblock_t.grad is not None and float(resblock_t.grad.norm()) > 0

## 5. Tiny U-Net：多尺度路径与 skip concatenation

U-Net 在高分辨率保留局部细节，在低分辨率扩大感受野。教学结构为：input conv → high-resolution ResBlock → stride-2 downsample → middle ResBlock → transposed-conv upsample → 与 high-resolution skip **拼接** → output ResBlock。

拼接使 up block 输入通道为两支通道之和；若误写成相加，结构可能仍能运行，却丢失一半特征表达。输出和输入 shape 必须完全一致，因为目标 epsilon 与 $x_t$ 同形。

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=8, time_embedding_dim=16, timesteps=32):
        super().__init__()
        self.in_channels, self.timesteps = int(in_channels), int(timesteps)
        self.base_channels = int(base_channels)
        self.time_embedding_dim = int(time_embedding_dim)
        time_dim = 2 * self.time_embedding_dim
        self.time_encoder = SinusoidalTimeEmbedding(self.time_embedding_dim)
        self.time_mlp = nn.Sequential(nn.Linear(self.time_embedding_dim, time_dim), nn.SiLU(),
                                      nn.Linear(time_dim, time_dim))
        self.input_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.high_block = TimeConditionedResBlock(base_channels, 2 * base_channels, time_dim)
        self.downsample = nn.Conv2d(2 * base_channels, 2 * base_channels, 4, stride=2, padding=1)
        self.middle = TimeConditionedResBlock(2 * base_channels, 2 * base_channels, time_dim)
        self.upsample = nn.ConvTranspose2d(2 * base_channels, 2 * base_channels, 4, stride=2, padding=1)
        self.up_block = TimeConditionedResBlock(4 * base_channels, base_channels, time_dim)
        self.output_norm = nn.GroupNorm(valid_groups(base_channels), base_channels)
        self.output_conv = nn.Conv2d(base_channels, in_channels, 3, padding=1)

    def forward(self, noisy_images, timesteps):
        if noisy_images.ndim != 4 or noisy_images.shape[1] != self.in_channels:
            raise ValueError("TinyUNet expects configured NCHW input")
        if timesteps.shape != (noisy_images.shape[0],) or timesteps.dtype != torch.long:
            raise ValueError("timesteps must be int64 [B]")
        if (timesteps < 0).any() or (timesteps >= self.timesteps).any():
            raise ValueError("timestep out of model range")
        time_embedding = self.time_mlp(self.time_encoder(timesteps))
        high = self.high_block(self.input_conv(noisy_images), time_embedding)
        middle = self.middle(self.downsample(high), time_embedding)
        up = self.upsample(middle)
        if up.shape[-2:] != high.shape[-2:]:
            raise ValueError("input spatial dimensions are incompatible with U-Net scale")
        fused = torch.cat([up, high], dim=1)
        return self.output_conv(F.silu(self.output_norm(self.up_block(fused, time_embedding))))

unet_probe = TinyUNet()
unet_input = torch.randn(3, 1, 8, 8, requires_grad=True)
unet_output = unet_probe(unet_input, torch.tensor([0, 7, 31]))
unet_output.square().mean().backward()
assert unet_output.shape == unet_input.shape
assert unet_input.grad is not None and torch.isfinite(unet_input.grad).all()
assert float(unet_input.grad.norm()) > 0
assert unet_probe.time_mlp[0].weight.grad is not None

## 6. `q_sample`：一次得到任意 $x_t$

训练不必真的循环加噪 $t$ 次。根据闭式公式，从 batch 中为每个样本抽不同 $t$ 和 $\epsilon$，一次构造 $x_t$。随机噪声必须能由调用方传入或通过显式 `torch.Generator` 生成，测试才可重放。

下面用手工给定的 $x_0,t,\epsilon$ 做数值 oracle；同时验证相同 generator seed 逐位相同、不同 seed 不同。全局 `manual_seed` 不是并发服务的请求级随机性合同。

In [ ]:
def extract_coefficient(values, timesteps, reference):
    if timesteps.dtype != torch.long or timesteps.shape != (reference.shape[0],):
        raise ValueError("coefficient timestep shape mismatch")
    if (timesteps < 0).any() or (timesteps >= len(values)).any():
        raise ValueError("coefficient timestep out of range")
    return values[timesteps].reshape(-1, *([1] * (reference.ndim - 1)))

def q_sample(x0, timesteps, schedule, noise=None, generator=None):
    if x0.ndim != 4:
        raise ValueError("q_sample expects NCHW")
    if noise is None:
        noise = torch.randn(x0.shape, dtype=x0.dtype, device=x0.device, generator=generator)
    if noise.shape != x0.shape:
        raise ValueError("noise shape mismatch")
    alpha_bar = extract_coefficient(schedule.alpha_bars, timesteps, x0)
    return alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise, noise

oracle_x0 = torch.tensor([[[[-1., 0.], [.5, 1.]]], [[[.2, -.3], [.4, -.5]]]])
oracle_t = torch.tensor([0, 7], dtype=torch.long)
oracle_noise = torch.tensor([[[[.1, -.2], [.3, -.4]]], [[[.5, .6], [-.7, .8]]]])
oracle_xt, returned_noise = q_sample(oracle_x0, oracle_t, schedule36, noise=oracle_noise)
oracle_alpha_bar = schedule36.alpha_bars[oracle_t].reshape(2, 1, 1, 1)
expected_xt = oracle_alpha_bar.sqrt() * oracle_x0 + (1 - oracle_alpha_bar).sqrt() * oracle_noise
assert torch.allclose(oracle_xt, expected_xt)
assert torch.equal(returned_noise, oracle_noise)

generator_a = torch.Generator().manual_seed(991)
generator_b = torch.Generator().manual_seed(991)
generator_c = torch.Generator().manual_seed(992)
sample_a = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_a)[0]
sample_b = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_b)[0]
sample_c = q_sample(torch.zeros(2, 1, 4, 4), oracle_t, schedule36, generator=generator_c)[0]
assert torch.equal(sample_a, sample_b)
assert not torch.equal(sample_a, sample_c)

## 7. epsilon prediction objective

原始 DDPM 常用简化目标

$$L_{simple}=\mathbb E_{x_0,t,\epsilon}\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2.$$

这是对随机 $t$、随机噪声的 Monte Carlo 估计。reduction 必须明确：这里对 batch、通道和空间全部取均值。若误把 target 写成 $x_0$，模型变成另一种 parameterization，反向公式也必须同时改变，不能混搭。

In [ ]:
def epsilon_prediction_loss(model, clean_images, schedule, generator):
    batch = clean_images.shape[0]
    timesteps = torch.randint(0, schedule.timesteps, (batch,), generator=generator,
                              device=clean_images.device)
    noisy_images, target_noise = q_sample(clean_images, timesteps, schedule, generator=generator)
    predicted_noise = model(noisy_images, timesteps)
    if predicted_noise.shape != target_noise.shape:
        raise ValueError("epsilon prediction shape mismatch")
    return F.mse_loss(predicted_noise, target_noise), timesteps, target_noise

loss_model_probe = TinyUNet()
loss_generator = torch.Generator().manual_seed(1701)
probe_loss, sampled_t, sampled_noise = epsilon_prediction_loss(
    loss_model_probe, torch.randn(5, 1, 8, 8), schedule36, loss_generator
)
probe_loss.backward()
assert probe_loss.ndim == 0 and torch.isfinite(probe_loss)
assert sampled_t.shape == (5,) and sampled_noise.shape == (5, 1, 8, 8)
assert loss_model_probe.output_conv.weight.grad is not None
assert float(loss_model_probe.output_conv.weight.grad.norm()) > 0

## 8. 受控 $8\times8$ 图案与数据范围

训练分布由竖条、横条、十字三种图案组成，位置和宽度变化，像素严格映射到 `[-1,1]`。扩散模型输出层没有 sigmoid/tanh；采样边界由 $x_0$ clipping 合同控制。

train 与 validation 使用不同 seed 的轻微噪声。真实图像应固定 resize/crop、颜色空间和从 `[0,255]` 到模型范围的映射，并把这些字段放入制品；否则 schedule 相同也不是同一个生成任务。

In [ ]:
def make_diffusion_patterns(count, seed):
    generator = torch.Generator().manual_seed(seed)
    images = -torch.ones(count, 1, 8, 8)
    for index in range(count):
        label, position = index % 3, 1 + (index // 3) % 6
        if label in (0, 2):
            images[index, 0, :, position:position + 1] = 1.0
        if label in (1, 2):
            images[index, 0, position:position + 1, :] = 1.0
    images = (images + 0.03 * torch.randn(images.shape, generator=generator)).clamp(-1, 1)
    return images

train36 = make_diffusion_patterns(36, SEED + 1)
valid36 = make_diffusion_patterns(18, SEED + 2)
assert train36.shape == (36, 1, 8, 8)
assert valid36.shape == (18, 1, 8, 8)
assert float(train36.min()) >= -1 and float(train36.max()) <= 1
assert not torch.equal(train36[:18], valid36)

## 9. 受控训练、独立随机流与 validation checkpoint

每步从 train 随机抽 mini-batch，训练 generator 独占自己的随机流。validation 固定一组 $(t,\epsilon)$，因此不同 checkpoint 的 denoising MSE 可公平比较；它不消费训练随机流。test/sample seed 也应另行分配。

第一步检查 time path、卷积路径和输出头梯度。使用 validation 选择 checkpoint，绝不根据最终生成样本手工挑选训练步数。小样本 loss 下降是计算图 smoke test，不是样本质量结论。

In [ ]:
torch.manual_seed(SEED)
model36 = TinyUNet().to(DEVICE)
optimizer = torch.optim.Adam(model36.parameters(), lr=0.004)
train_generator = torch.Generator().manual_seed(SEED + 100)
validation_generator = torch.Generator().manual_seed(SEED + 200)
validation_t = torch.randint(0, schedule36.timesteps, (len(valid36),), generator=validation_generator)
validation_noise = torch.randn(valid36.shape, generator=validation_generator)
validation_xt, _ = q_sample(valid36, validation_t, schedule36, noise=validation_noise)
history, validation_history = [], []
best_validation, best_state = float("inf"), None

for step in range(121):
    indices = torch.randint(0, len(train36), (12,), generator=train_generator)
    clean_batch = train36[indices]
    model36.train()
    optimizer.zero_grad(set_to_none=True)
    loss, _, _ = epsilon_prediction_loss(model36, clean_batch, schedule36, train_generator)
    loss.backward()
    if step == 0:
        gradient_checks = {
            "input": model36.input_conv.weight.grad.norm(),
            "time": model36.time_mlp[0].weight.grad.norm(),
            "middle": model36.middle.conv1.weight.grad.norm(),
            "output": model36.output_conv.weight.grad.norm(),
        }
    torch.nn.utils.clip_grad_norm_(model36.parameters(), 1.0)
    optimizer.step()
    history.append(float(loss.detach()))
    if step % 10 == 0:
        model36.eval()
        with torch.no_grad():
            validation_mse = float(F.mse_loss(model36(validation_xt, validation_t), validation_noise))
        validation_history.append((step, validation_mse))
        if validation_mse < best_validation:
            best_validation, best_state = validation_mse, deepcopy(model36.state_dict())

assert best_state is not None
model36.load_state_dict(best_state)
assert all(torch.isfinite(value) and float(value) > 0 for value in gradient_checks.values())
assert sum(history[-20:]) / 20 < sum(history[:20]) / 20
assert best_validation < float(validation_noise.square().mean())
print({"train_mse_first20_last20": [sum(history[:20]) / 20, sum(history[-20:]) / 20],
       "best_validation_epsilon_mse": best_validation,
       "zero_predictor_validation_mse": float(validation_noise.square().mean())})

## 10. 正确反向 posterior：先预测并 clip $x_0$

由 epsilon 预测恢复

$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

若启用 clipping，先把 $\hat x_0$ 截到训练数据范围，再用真实后验均值系数

$$\tilde\mu_t=c_1(t)\hat x_0+c_2(t)x_t$$

重算均值。若只 clip 最终 `x_{t-1}`，就不再对应这个 posterior。$t>0$ 添加方差 $\tilde\beta_t$ 的噪声；$t=0$ 返回均值，不消费噪声。

In [ ]:
def predict_x0_from_epsilon(x_t, timesteps, predicted_noise, schedule, clip=True):
    alpha_bar = extract_coefficient(schedule.alpha_bars, timesteps, x_t)
    x0 = (x_t - (1 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()
    return x0.clamp(-1.0, 1.0) if clip else x0

@torch.no_grad()
def p_sample(model, x_t, timesteps, schedule, generator, clip_x0=True):
    if timesteps.shape != (x_t.shape[0],):
        raise ValueError("reverse timestep shape mismatch")
    predicted_noise = model(x_t, timesteps)
    predicted_x0 = predict_x0_from_epsilon(x_t, timesteps, predicted_noise, schedule, clip_x0)
    coef1 = extract_coefficient(schedule.posterior_mean_coef1, timesteps, x_t)
    coef2 = extract_coefficient(schedule.posterior_mean_coef2, timesteps, x_t)
    posterior_mean = coef1 * predicted_x0 + coef2 * x_t
    variance = extract_coefficient(schedule.posterior_variance, timesteps, x_t)
    active = timesteps > 0
    noise = torch.zeros_like(x_t)
    if active.any():
        active_shape = (int(active.sum().item()), *x_t.shape[1:])
        noise[active] = torch.randn(active_shape, dtype=x_t.dtype, device=x_t.device,
                                    generator=generator)
    previous = posterior_mean + variance.sqrt() * noise
    return previous, predicted_x0, posterior_mean

class ZeroEpsilon(nn.Module):
    def forward(self, x, timesteps):
        return torch.zeros_like(x)

zero_model = ZeroEpsilon()
reverse_x = torch.full((2, 1, 2, 2), 0.25)
reverse_t = torch.tensor([0, 5], dtype=torch.long)
reverse_generator = torch.Generator().manual_seed(881)
reverse_result, reverse_x0, reverse_mean = p_sample(
    zero_model, reverse_x, reverse_t, schedule36, reverse_generator
)
expected_x0 = predict_x0_from_epsilon(reverse_x, reverse_t, torch.zeros_like(reverse_x), schedule36)
expected_mean = (extract_coefficient(schedule36.posterior_mean_coef1, reverse_t, reverse_x) * expected_x0 +
                 extract_coefficient(schedule36.posterior_mean_coef2, reverse_t, reverse_x) * reverse_x)
replay_noise = torch.zeros_like(reverse_x)
replay_noise[1:] = torch.randn((1, 1, 2, 2), generator=torch.Generator().manual_seed(881))
expected_previous = expected_mean + extract_coefficient(
    schedule36.posterior_variance, reverse_t, reverse_x
).sqrt() * replay_noise
assert torch.allclose(reverse_x0, expected_x0)
assert torch.allclose(reverse_mean, expected_mean)
assert torch.allclose(reverse_result, expected_previous)
assert torch.equal(reverse_result[0], reverse_mean[0])  # t=0 无随机项
zero_only_generator = torch.Generator().manual_seed(1234)
generator_state_before = zero_only_generator.get_state().clone()
_ = p_sample(zero_model, reverse_x[:1], torch.tensor([0]), schedule36, zero_only_generator)
assert torch.equal(generator_state_before, zero_only_generator.get_state())  # t=0 不消费随机流

large_xt = torch.full((1, 1, 2, 2), 20.0)
last_t = torch.tensor([31])
assert predict_x0_from_epsilon(large_xt, last_t, torch.zeros_like(large_xt), schedule36, False).max() > 1
assert predict_x0_from_epsilon(large_xt, last_t, torch.zeros_like(large_xt), schedule36, True).max() == 1

## 11. 完整 DDPM 采样与确定性 oracle

采样函数拥有一个请求级 generator：它先生成初始 $x_T$，再按顺序生成每一步 posterior noise。同一 seed、同一模型、同一 schedule 应逐位复现；不同 seed 应产生不同结果。若在循环内部反复把 generator 重置到同一 seed，会让每一步噪声异常相关。

最终 $t=0$ 使用 clipped $\hat x_0$，所以输出应在 `[-1,1]`。这项范围断言只检查接口合同，不评价样本是否像训练数据。

In [ ]:
@torch.no_grad()
def sample_ddpm(model, schedule, sample_shape, seed):
    if len(sample_shape) != 4 or sample_shape[1] != model.in_channels:
        raise ValueError("sample shape mismatch")
    if not 1 <= sample_shape[0] <= 32:
        raise ValueError("sample batch out of bounds")
    model.eval()
    device = next(model.parameters()).device
    generator = torch.Generator(device=device).manual_seed(int(seed))
    current = torch.randn(sample_shape, generator=generator, device=device)
    for timestep in reversed(range(schedule.timesteps)):
        t = torch.full((sample_shape[0],), timestep, dtype=torch.long, device=device)
        current, _, _ = p_sample(model, current, t, schedule, generator, clip_x0=True)
    return current.cpu()

samples_a = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7001)
samples_b = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7001)
samples_c = sample_ddpm(model36, schedule36, (4, 1, 8, 8), seed=7002)
assert torch.equal(samples_a, samples_b)
assert not torch.equal(samples_a, samples_c)
assert samples_a.shape == (4, 1, 8, 8)
assert float(samples_a.min()) >= -1.0 and float(samples_a.max()) <= 1.0
assert float((samples_a[0] - samples_a[1]).abs().mean()) > 1e-4
print({"sample_min_max": [float(samples_a.min()), float(samples_a.max())],
       "sample_mean_std": [float(samples_a.mean()), float(samples_a.std())]})

## 12. 评估：denoising 指标与生成质量不是一回事

固定 validation $(x_t,t,\epsilon)$ 上的 epsilon MSE 可诊断训练和 checkpoint，但较低 MSE 不必然产生更好的样本。受控例再比较生成图的像素范围与多样性；这些也不能替代分布质量评估。

真实图像常报告 FID/KID、precision/recall、重复记忆与最近邻、条件一致性和人评，并给出置信区间。FID 对样本数和 feature extractor 敏感，不能在几十张微型样本上解释为质量结论。

In [ ]:
model36.eval()
with torch.no_grad():
    validation_prediction = model36(validation_xt, validation_t)
    validation_mse = float(F.mse_loss(validation_prediction, validation_noise))
    zero_baseline_mse = float(validation_noise.square().mean())
    reconstructed_x0 = predict_x0_from_epsilon(validation_xt, validation_t,
                                                validation_prediction, schedule36, clip=True)

assert math.isclose(validation_mse, best_validation, rel_tol=0, abs_tol=1e-7)
assert validation_mse < zero_baseline_mse
assert ((reconstructed_x0 >= -1) & (reconstructed_x0 <= 1)).all()
pairwise_sample_distance = torch.pdist(samples_a.flatten(1)).mean()
assert torch.isfinite(pairwise_sample_distance) and float(pairwise_sample_distance) > 0
print({"fixed_validation_epsilon_mse": validation_mse,
       "zero_predictor_mse": zero_baseline_mse,
       "generated_pairwise_l2": float(pairwise_sample_distance)})

## 13. 制品合同：真实模型配置、schedule、数据快照与外部信任锚

扩散模型的权重不能表达训练时的 beta schedule、epsilon parameterization 或输入映射。builder 必须从真实 `TinyUNet` 与 `DiffusionSchedule` 实例导出 `in_channels/base_channels/time_embedding_dim/timesteps/beta_*`，并先拒绝 model 与 schedule 的步数不一致；硬编码 32 会产生“模型 32、schedule 16 仍能加载”的静默错误。

package 内部 hash 只负责完整性，不是身份认证。这里另外使用发布者侧只读登记 `artifact_id/version -> expected bundle digest`；bundle 对 canonical manifest 与 state 每个 tensor 的 `key/dtype/shape/bytes` 一起做长度分隔哈希。train/validation 图像 split、固定 validation 的 `(t, epsilon)` target、随机流 recipe 和输入 `[-1,1]` 预处理也全部绑定并可重建。整体替换后重签所有内部摘要、伪造 input shape，仍无法改变 package 外的登记值。生产环境应把这一登记落到签名发布元数据或只读制品服务。

In [ ]:
def canonical_json36(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False).encode("utf-8")

def _feed_digest36(hasher, payload):
    hasher.update(len(payload).to_bytes(8, "big"))
    hasher.update(payload)

def clone_state36(state_dict):
    if not hasattr(state_dict, "items"):
        raise ValueError("state_dict must be a mapping")
    cloned = {}
    for key, tensor in state_dict.items():
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("state entries must be string -> Tensor")
        cloned[key] = tensor.detach().cpu().contiguous().clone()
    return cloned

def _update_state_digest36(hasher, state_dict):
    if not isinstance(state_dict, dict) or not state_dict:
        raise ValueError("artifact state_dict must be a non-empty plain dict")
    for key in sorted(state_dict):
        tensor = state_dict[key]
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("invalid state entry")
        cpu = tensor.detach().cpu().contiguous()
        header = canonical_json36({"key": key, "dtype": str(cpu.dtype),
                                   "shape": list(cpu.shape)})
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()
        _feed_digest36(hasher, header)
        _feed_digest36(hasher, raw)

def canonical_state_digest36(state_dict):
    hasher = sha256()
    _feed_digest36(hasher, b"canonical-state-dict-v1")
    _update_state_digest36(hasher, state_dict)
    return hasher.hexdigest()

def canonical_bundle_digest36(manifest, state_dict):
    hasher = sha256()
    _feed_digest36(hasher, b"canonical-model-bundle-v1")
    _feed_digest36(hasher, canonical_json36(manifest))
    _update_state_digest36(hasher, state_dict)
    return hasher.hexdigest()

def state_schema36(state_dict):
    return [{"key": key, "dtype": str(state_dict[key].dtype),
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]

def expected_diffusion_data36(timesteps):
    split_specs = {"train": (36, SEED + 1), "validation": (18, SEED + 2)}
    splits = {}
    for name, (count, seed) in split_specs.items():
        images = make_diffusion_patterns(count, seed)
        splits[name] = {"count": count, "seed": seed,
                        "images_sha256": canonical_state_digest36(
                            clone_state36({"images": images}))}
    validation_generator_replay = torch.Generator().manual_seed(SEED + 200)
    replay_t = torch.randint(0, timesteps, (18,), generator=validation_generator_replay)
    replay_noise = torch.randn((18, 1, 8, 8), generator=validation_generator_replay)
    validation_target_digest = canonical_state_digest36(
        clone_state36({"timesteps": replay_t, "epsilon_target": replay_noise}))
    return {
        "dataset_recipe": "controlled-8x8-patterns-noise-clamp-v1", "splits": splits,
        "training_target": "epsilon", "timestep_sampling": "discrete-uniform-[0,T)",
        "noise_distribution": "standard-normal", "train_generator_seed": SEED + 100,
        "validation_generator_seed": SEED + 200,
        "validation_t_epsilon_sha256": validation_target_digest,
    }

def schedule_contract36(schedule):
    return {"kind": "linear", "timesteps": int(schedule.timesteps),
            "beta_start": float(schedule.beta_start), "beta_end": float(schedule.beta_end),
            "betas_digest_sha256": canonical_state_digest36(
                clone_state36({"betas": schedule.betas}))}

EXPECTED_MODEL_CONFIG36 = {"in_channels": 1, "base_channels": 8,
                           "time_embedding_dim": 16, "timesteps": 32}
EXPECTED_PREPROCESS36 = {"input_shape": [1, 8, 8], "layout": "NCHW",
                         "dtype": "float32", "data_range": [-1.0, 1.0],
                         "recipe": "controlled-pattern-noise-then-clamp-v1"}
ARTIFACT_ID36, ARTIFACT_VERSION36 = "vision.controlled-ddpm", "1.0.0"

def build_diffusion_artifact(model, schedule):
    if type(model) is not TinyUNet or type(schedule) is not DiffusionSchedule:
        raise ValueError("publisher only accepts audited TinyUNet and DiffusionSchedule")
    model_config = {"in_channels": model.in_channels, "base_channels": model.base_channels,
                    "time_embedding_dim": model.time_embedding_dim,
                    "timesteps": model.timesteps}
    if model_config["timesteps"] != schedule.timesteps:
        raise ValueError("model and schedule timesteps must match before publishing")
    state = clone_state36(model.state_dict())
    manifest = {
        "schema_version": 2, "artifact_id": ARTIFACT_ID36,
        "artifact_version": ARTIFACT_VERSION36, "architecture": "TinyUNet",
        "model_config": model_config, "model_state_schema": state_schema36(state),
        "schedule": schedule_contract36(schedule),
        "preprocess": deepcopy(EXPECTED_PREPROCESS36),
        "prediction_parameterization": "epsilon",
        "data_contract": expected_diffusion_data36(schedule.timesteps),
        "state_digest_sha256": canonical_state_digest36(state),
    }
    return {"manifest": manifest,
            "manifest_sha256": sha256(canonical_json36(manifest)).hexdigest(),
            "bundle_sha256": canonical_bundle_digest36(manifest, state),
            "state_dict": state}

def validate_diffusion_contract36(manifest, state_dict):
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",
                "model_config", "model_state_schema", "schedule", "preprocess",
                "prediction_parameterization", "data_contract", "state_digest_sha256"}
    if set(manifest) != required or manifest["schema_version"] != 2:
        raise ValueError("manifest schema mismatch")
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID36, ARTIFACT_VERSION36):
        raise ValueError("artifact identity mismatch")
    if manifest["architecture"] != "TinyUNet" or manifest["prediction_parameterization"] != "epsilon":
        raise ValueError("unsupported diffusion architecture/parameterization")
    model_config, schedule_config = manifest["model_config"], manifest["schedule"]
    if model_config.get("timesteps") != schedule_config.get("timesteps"):
        raise ValueError("model/schedule timestep mismatch")
    if model_config != EXPECTED_MODEL_CONFIG36:
        raise ValueError("model config mismatch")
    if manifest["preprocess"] != EXPECTED_PREPROCESS36:
        raise ValueError("input shape/preprocess mismatch")
    expected_schedule = schedule_contract36(DiffusionSchedule(32, 1e-4, 0.18))
    if schedule_config != expected_schedule:
        raise ValueError("schedule config or beta tensor mismatch")
    if manifest["data_contract"] != expected_diffusion_data36(32):
        raise ValueError("train/validation snapshot or randomness recipe mismatch")
    expected_schema = state_schema36(TinyUNet(**EXPECTED_MODEL_CONFIG36).state_dict())
    if manifest["model_state_schema"] != expected_schema or state_schema36(state_dict) != expected_schema:
        raise ValueError("model state schema mismatch")

def load_trusted_diffusion(artifact):
    if not isinstance(artifact, dict) or set(artifact) != {
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:
        raise ValueError("artifact package schema mismatch")
    manifest, state = artifact["manifest"], artifact["state_dict"]
    if not isinstance(manifest, dict):
        raise ValueError("manifest must be a dict")
    actual_bundle = canonical_bundle_digest36(manifest, state)
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))
    expected_bundle = PUBLISHER_REGISTRY36.get(identity)
    if expected_bundle is None or actual_bundle != expected_bundle:
        raise ValueError("publisher registry rejected this bundle")
    if artifact["bundle_sha256"] != actual_bundle:
        raise ValueError("internal bundle digest mismatch")
    if sha256(canonical_json36(manifest)).hexdigest() != artifact["manifest_sha256"]:
        raise ValueError("manifest digest mismatch")
    if canonical_state_digest36(state) != manifest["state_digest_sha256"]:
        raise ValueError("canonical state digest mismatch")
    validate_diffusion_contract36(manifest, state)
    schedule_config = manifest["schedule"]
    loaded_schedule = DiffusionSchedule(schedule_config["timesteps"],
                                        schedule_config["beta_start"], schedule_config["beta_end"])
    loaded_model = TinyUNet(**manifest["model_config"])
    loaded_model.load_state_dict(state, strict=True)
    return loaded_model.eval(), loaded_schedule

def resign_inside36(artifact):
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest36(artifact["state_dict"])
    artifact["manifest_sha256"] = sha256(canonical_json36(artifact["manifest"])).hexdigest()
    artifact["bundle_sha256"] = canonical_bundle_digest36(artifact["manifest"], artifact["state_dict"])
    return artifact

artifact36 = build_diffusion_artifact(model36, schedule36)
PUBLISHER_REGISTRY36 = MappingProxyType({
    (ARTIFACT_ID36, ARTIFACT_VERSION36): artifact36["bundle_sha256"]
})
loaded_model36, loaded_schedule36 = load_trusted_diffusion(artifact36)
loaded_sample36 = sample_ddpm(loaded_model36, loaded_schedule36, (2, 1, 8, 8), seed=333)
original_sample36 = sample_ddpm(model36, schedule36, (2, 1, 8, 8), seed=333)
assert torch.equal(loaded_sample36, original_sample36)

# builder 必须读取真实实例：16-step 模型与 16-step schedule 的 manifest 都是 16，不再硬编码 32。
short_model36 = TinyUNet(timesteps=16)
short_schedule36 = DiffusionSchedule(timesteps=16, beta_start=1e-4, beta_end=0.10)
short_artifact36 = build_diffusion_artifact(short_model36, short_schedule36)
assert short_artifact36["manifest"]["model_config"]["timesteps"] == 16
assert short_artifact36["manifest"]["schedule"]["timesteps"] == 16
assert short_artifact36["manifest"]["schedule"]["beta_end"] == 0.10
try:
    load_trusted_diffusion(short_artifact36)
    raise AssertionError("unpublished self-signed whole replacement must fail")
except ValueError:
    pass

# 发布前和加载语义校验都拒绝 model=16 / schedule=32。
try:
    build_diffusion_artifact(short_model36, schedule36)
    raise AssertionError("builder must reject model/schedule timestep mismatch")
except ValueError:
    pass
mismatch36 = deepcopy(artifact36)
mismatch36["manifest"]["model_config"]["timesteps"] = 16
resign_inside36(mismatch36)
try:
    validate_diffusion_contract36(mismatch36["manifest"], mismatch36["state_dict"])
    raise AssertionError("semantic validator must reject 16/32 mismatch")
except ValueError:
    pass
try:
    load_trusted_diffusion(mismatch36)
    raise AssertionError("self-signed 16/32 mismatch must fail")
except ValueError:
    pass

wrong_shape36 = deepcopy(artifact36)
wrong_shape36["manifest"]["preprocess"]["input_shape"] = [1, 16, 16]
resign_inside36(wrong_shape36)
try:
    validate_diffusion_contract36(wrong_shape36["manifest"], wrong_shape36["state_dict"])
    raise AssertionError("semantic validator must reject forged input shape")
except ValueError:
    pass
try:
    load_trusted_diffusion(wrong_shape36)
    raise AssertionError("self-signed input shape replacement must fail")
except ValueError:
    pass

try:
    PUBLISHER_REGISTRY36[(ARTIFACT_ID36, ARTIFACT_VERSION36)] = short_artifact36["bundle_sha256"]
    raise AssertionError("publisher registry must be immutable")
except TypeError:
    pass

## 14. 失败模式、复杂度与生产差距

1. **训练预测 epsilon，采样却按 x0 参数化解释**：公式不配套，采样会崩坏。
2. **posterior variance 直接使用 $\beta_t$**：真实 $q(x_{t-1}\mid x_t,x_0)$ 方差是 $\tilde\beta_t$；尤其 $t=0$ 必须为零。
3. **clip 错位置**：应 clip 预测 $x_0$ 后重算 posterior mean，而不是只截断 noisy $x_{t-1}$。
4. **循环内重置 seed**：每步噪声高度相关；一个请求只创建一次 generator 并顺序消费。
5. **训练/validation 共用随机流**：改变评估频率会改变后续训练轨迹；必须拆分 generator。
6. **schedule 或数据 recipe 未由外部发布摘要绑定**：自签内部 hash 仍可整体替换并造成语义错位。

每个 U-Net step 的卷积成本约随 $O(HWC^2)$ 增长，DDPM 总采样成本再乘 $T$。现代生产还需更大的 U-Net/DiT、attention、EMA、mixed precision 数值审计、分布式训练、classifier-free guidance、快速 sampler、内容安全、版权/隐私/记忆审计和目标硬件 p95/p99 压测。

### 论文来源

- Ho, Jain, Abbeel, [*Denoising Diffusion Probabilistic Models*](https://arxiv.org/abs/2006.11239), NeurIPS 2020.
- Sohl-Dickstein et al., [*Deep Unsupervised Learning using Nonequilibrium Thermodynamics*](https://arxiv.org/abs/1503.03585), ICML 2015.
- Ronneberger, Fischer, Brox, [*U-Net: Convolutional Networks for Biomedical Image Segmentation*](https://arxiv.org/abs/1505.04597), MICCAI 2015（多尺度 skip 架构背景）。

本册复现 DDPM 的核心离散公式与微型网络，不声称复现论文数据规模、采样速度或图像质量。